# Notebook 7: Q-Learning (Tabular)

In previous notebooks, we trained policies using **policy-based methods** (random search, hill climbing, evolutionary strategies). These methods treat the policy as a black box and optimize it using feedback from complete episodes.

In this notebook, we explore **value-based learning** — specifically **Q-learning**, one of the foundational algorithms in reinforcement learning.

## Learning Objectives

By the end of this notebook, you will understand:
1. The difference between policy-based and value-based RL
2. What a Q-function (action-value function) is and why it matters
3. The Bellman equation and the TD update rule
4. State space discretization (binning) and its trade-offs
5. Epsilon-greedy exploration and epsilon decay schedules
6. How tabular Q-learning compares to evolutionary methods on our pendulum

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.environments import InvertedPendulumEnv
from src.policies import RandomPolicy, LinearPolicy, QPolicy
from src.utils import evaluate_policy, collect_episode, train_policy
from src.utils.training import train_q_learning

print('Imports OK')

## Section 1: Value-Based vs Policy-Based RL

| Approach | What it learns | Update frequency | Examples |
|---|---|---|---|
| **Policy-based** | Direct mapping: state → action | Per episode | Random Search, Evolution Strategies |
| **Value-based** | Value function: (state, action) → expected return | Per step | Q-Learning, DQN |

### The Q-function

The **Q-function** (action-value function) `Q(s, a)` answers the question:

> "If I'm in state `s` and take action `a`, what total discounted reward should I expect to get?"

Once we have Q-values, the optimal policy is simply:
```
π*(s) = argmax_a Q*(s, a)
```
Always pick the action with the highest Q-value — no separate policy parameters needed!

### The Bellman Equation

Q-learning is grounded in the **Bellman optimality equation**:

```
Q*(s, a) = E[r + γ · max_a' Q*(s', a')]
```

In plain English: the value of taking action `a` in state `s` equals the immediate reward `r` plus the discounted value of the best action in the next state `s'`.

### The Q-Learning Update (TD Update)

We can't directly compute this expectation, so Q-learning uses a **Temporal Difference (TD) update** that incrementally improves Q-values using observed transitions:

```
Q(s, a) ← Q(s, a) + α · [r + γ · max_a' Q(s', a') - Q(s, a)]
                                └────── TD target ──────┘   └── TD error ──┘
```

- `α` (alpha): Learning rate — how fast to update Q-values
- `γ` (gamma): Discount factor — how much to value future rewards
- **TD error**: The "surprise" — how far our current estimate is from the bootstrapped target

## Section 2: State Space Discretization

Q-learning with a lookup table requires **discrete** states. Our inverted pendulum has a continuous 4D state:
```
state = [x, x_dot, theta, theta_dot]
```

We discretize each dimension into bins. More bins = better precision, but exponentially larger Q-table (the **curse of dimensionality**).

Our default configuration:
- `x`: 6 bins in [-2.4, 2.4] m
- `x_dot`: 6 bins in [-3.0, 3.0] m/s (clipped)
- `theta`: 10 bins in [-0.2095, 0.2095] rad — most critical!
- `theta_dot`: 10 bins in [-3.0, 3.0] rad/s (clipped)

Total states: 6 × 6 × 10 × 10 = **3,600** discrete states

In [ ]:
env = InvertedPendulumEnv()
policy = QPolicy(n_actions=3, alpha=0.1, gamma=0.99, epsilon=1.0)

print(f'Q-table shape: {policy.q_table.shape}')
print(f'Q-table total entries: {policy.q_table.size:,}')
print(f'Discrete action values: {policy._action_values}')
print()

# Visualize the bin structure
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
state_names = ['x (pos)', 'x_dot (vel)', 'theta (angle)', 'theta_dot (angular vel)']
colors = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F']

for i, (ax, name, color, (low, high, n_bins)) in enumerate(
    zip(axes, state_names, colors, policy.state_bins)
):
    edges = np.linspace(low, high, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    ax.bar(centers, [1] * n_bins, width=(high - low) / n_bins * 0.9,
           color=color, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{name}\n({n_bins} bins)', fontsize=9)
    ax.set_xlabel(f'[{low:.2f}, {high:.2f}]', fontsize=8)
    ax.set_yticks([])

plt.suptitle('State Space Discretization', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 3: Epsilon-Greedy Exploration

Q-learning faces the **exploration-exploitation dilemma**:
- **Explore**: Try random actions to discover new states and update Q-values
- **Exploit**: Use current Q-values to maximize reward

The **epsilon-greedy** strategy handles this:
- With probability `ε`: random action (explore)
- With probability `1-ε`: argmax Q-value action (exploit)

We start with high `ε` (pure exploration) and decay it over time.

In [ ]:
from src.utils.training import _compute_epsilon

n_episodes = 2000
episodes = np.arange(n_episodes)

schedules = {
    'linear': [_compute_epsilon(ep, n_episodes, 1.0, 0.01, 'linear') for ep in episodes],
    'exponential': [_compute_epsilon(ep, n_episodes, 1.0, 0.01, 'exponential') for ep in episodes],
    'cosine': [_compute_epsilon(ep, n_episodes, 1.0, 0.01, 'cosine') for ep in episodes],
}

fig, ax = plt.subplots(figsize=(10, 4))
for name, epsilons in schedules.items():
    ax.plot(episodes, epsilons, label=name, linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon (ε)')
ax.set_title('Epsilon Decay Schedules')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0.01, color='gray', linestyle='--', alpha=0.5, label='epsilon_end')
plt.tight_layout()
plt.show()

## Section 4: Training Q-Learning

In [ ]:
env = InvertedPendulumEnv()
policy = QPolicy(
    n_actions=3,
    alpha=0.1,
    gamma=0.99,
    epsilon=1.0,
    seed=42,
)

print('Training Q-Learning...')
result = train_q_learning(
    env,
    policy,
    n_episodes=2000,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_schedule='exponential',
    eval_interval=100,
    n_eval_episodes=20,
    verbose=True,
    seed=42,
)

print(f'\nBest evaluation reward: {result["best_eval_reward"]:.1f}')
print(f'Final epsilon: {policy.epsilon:.4f}')
print(f'Q-table visited fraction: {policy.get_visited_fraction():.1%}')
print(f'Total Q-table updates: {policy._n_updates:,}')

In [ ]:
# Training progress visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Episode rewards with moving average
ax = axes[0, 0]
rewards = result['episode_rewards']
ax.plot(rewards, alpha=0.2, color='steelblue', label='Episode Reward')
window = 100
if len(rewards) >= window:
    ma = np.convolve(rewards, np.ones(window) / window, mode='valid')
    ax.plot(range(window - 1, len(rewards)), ma, color='steelblue',
            label=f'{window}-ep Moving Avg', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.set_title('Training Rewards')
ax.legend()
ax.grid(True, alpha=0.3)

# Evaluation rewards (greedy policy)
ax = axes[0, 1]
ax.plot(result['eval_episodes'], result['eval_rewards'], 'o-', color='green', linewidth=2)
ax.axhline(y=env.max_steps, color='gray', linestyle='--', label='Max possible')
ax.set_xlabel('Episode')
ax.set_ylabel('Mean Reward (greedy, ε=0)')
ax.set_title('Evaluation Rewards')
ax.legend()
ax.grid(True, alpha=0.3)

# Epsilon decay
ax = axes[1, 0]
ax.plot(result['epsilon_history'], color='orange', linewidth=1.5)
ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon (ε)')
ax.set_title('Epsilon Decay')
ax.grid(True, alpha=0.3)

# TD error (convergence indicator)
ax = axes[1, 1]
td_errors = result['td_error_history']
ax.plot(td_errors, alpha=0.3, color='red')
window = 100
if len(td_errors) >= window:
    ma = np.convolve(td_errors, np.ones(window) / window, mode='valid')
    ax.plot(range(window - 1, len(td_errors)), ma, color='red', linewidth=2,
            label=f'{window}-ep Moving Avg')
ax.set_xlabel('Episode')
ax.set_ylabel('Mean |TD Error|')
ax.set_title('TD Error (Convergence Indicator)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Q-Learning Training Progress', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5: Visualizing the Q-Table

Unlike neural network weights, Q-table values are **interpretable**. Let's look at what the agent learned by examining Q-values across states.

We'll fix `x=0` (cart centered) and `x_dot=0` (cart stationary), and visualize Q-values across `theta` × `theta_dot` — the most critical dimensions for balancing.

In [ ]:
action_names = ['Left\n(-10N)', 'No force\n(0N)', 'Right\n(+10N)']
n_theta_bins = policy.state_bins[2][2]     # 10
n_thetadot_bins = policy.state_bins[3][2]  # 10

# Fix x=0 (bin 3) and x_dot=0 (bin 3)
x_bin = 3
xdot_bin = 3

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

theta_range = np.linspace(*policy.state_bins[2][:2], n_theta_bins)
thetadot_range = np.linspace(*policy.state_bins[3][:2], n_thetadot_bins)

for a_idx, (ax, a_name) in enumerate(zip(axes, action_names)):
    q_matrix = policy.q_table[x_bin, xdot_bin, :, :, a_idx]
    # Shape: (n_theta_bins, n_thetadot_bins) -> display as (thetadot x theta)
    q_display = q_matrix.T

    vmax = np.abs(policy.q_table).max()
    im = ax.imshow(
        q_display,
        aspect='auto',
        cmap='RdYlGn',
        origin='lower',
        vmin=-vmax,
        vmax=vmax,
        extent=[
            policy.state_bins[2][0], policy.state_bins[2][1],
            policy.state_bins[3][0], policy.state_bins[3][1]
        ]
    )
    ax.set_xlabel('Theta (rad) — pole angle')
    ax.set_ylabel('Theta_dot (rad/s)' if a_idx == 0 else '')
    ax.set_title(f'Q-values: {a_name}')
    ax.axvline(x=0, color='white', linestyle='--', alpha=0.5)
    ax.axhline(y=0, color='white', linestyle='--', alpha=0.5)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Q-Table: Action Values at x=0, x_dot=0\n'
             '(Green = high value / preferred, Red = low value / avoided)',
             fontsize=11)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  - When theta > 0 (pole leaning right), pushing Right has LOW value')
print('    because it would make the pole fall further')
print('  - The agent should learn to push Left when theta > 0 and Right when theta < 0')

In [ ]:
# Demonstrate a greedy episode
policy.epsilon = 0.0  # Pure greedy
states, actions, rewards, _ = collect_episode(env, policy)

print(f'Greedy episode length: {len(rewards)} / {env.max_steps} steps')
print(f'Total reward: {sum(rewards):.0f}')

# Plot the trajectory
states_arr = np.array(states[:-1])  # exclude final state
actions_arr = np.array(actions)

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
t = np.arange(len(actions))
labels = ['x (cart pos)', 'x_dot (cart vel)', 'theta (angle)', 'theta_dot (ang vel)']

for i, (ax, label) in enumerate(zip(axes.flat, labels)):
    ax.plot(t, states_arr[:, i], linewidth=1.5)
    ax.set_xlabel('Step')
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

plt.suptitle('Greedy Q-Learning Policy — State Trajectory', fontsize=12)
plt.tight_layout()
plt.show()

## Section 6: Comparison with Existing Policies

Let's compare Q-learning against:
- **Random Policy**: Our baseline
- **Linear Policy (Evolutionary)**: Best policy-based approach from earlier notebooks

In [ ]:
eval_env = InvertedPendulumEnv()
N_EVAL = 50

# Random Policy baseline
random_policy = RandomPolicy(action_high=10.0, discrete=True, seed=42)
random_result = evaluate_policy(eval_env, random_policy, n_episodes=N_EVAL)

# Linear Policy (trained with evolutionary strategy)
print('Training Linear Policy (evolutionary)...')
linear_policy = LinearPolicy()
train_policy(
    eval_env, linear_policy,
    algorithm='evolutionary',
    n_iterations=100,
    population_size=20,
    verbose=False,
    seed=42,
)
linear_result = evaluate_policy(eval_env, linear_policy, n_episodes=N_EVAL)

# Q-Learning Policy (already trained; policy.epsilon = 0.0 set above)
q_result = evaluate_policy(eval_env, policy, n_episodes=N_EVAL)

print(f'\nResults over {N_EVAL} episodes:')
print(f'{"Policy":<20} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 56)
for name, res in [('Random', random_result), ('Linear (Evo)', linear_result), ('Q-Learning', q_result)]:
    ep_r = res['episode_rewards']
    print(f'{name:<20} {res["mean_reward"]:>8.1f} {res["std_reward"]:>8.1f} '
          f'{min(ep_r):>8.0f} {max(ep_r):>8.0f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

policy_names = ['Random\n(baseline)', 'Linear\n(Evolutionary)', 'Q-Learning\n(Tabular)']
means = [random_result['mean_reward'], linear_result['mean_reward'], q_result['mean_reward']]
stds  = [random_result['std_reward'],  linear_result['std_reward'],  q_result['std_reward']]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

bars = ax1.bar(policy_names, means, yerr=stds, capsize=8,
               color=colors, edgecolor='black', linewidth=0.8)
ax1.axhline(y=eval_env.max_steps, color='gray', linestyle='--', alpha=0.7, label=f'Max ({eval_env.max_steps})')
ax1.set_ylabel('Mean Episode Reward')
ax1.set_title('Policy Comparison — Mean Reward')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, mean in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             f'{mean:.0f}', ha='center', va='bottom', fontweight='bold')

# Reward distribution box plots
all_rewards = [
    random_result['episode_rewards'],
    linear_result['episode_rewards'],
    q_result['episode_rewards'],
]
bp = ax2.boxplot(all_rewards, labels=policy_names, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.axhline(y=eval_env.max_steps, color='gray', linestyle='--', alpha=0.7)
ax2.set_ylabel('Episode Reward')
ax2.set_title('Reward Distribution')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Policy Comparison on Inverted Pendulum', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 7: Hyperparameter Sensitivity

Let's explore how the number of bins affects performance. More bins = more precision but larger Q-table and slower learning.

In [ ]:
bin_configs = {
    'Coarse (3×3×4×4)': [[-2.4, 2.4, 3], [-3.0, 3.0, 3], [-0.2095, 0.2095, 4], [-3.0, 3.0, 4]],
    'Default (6×6×10×10)': None,  # uses QPolicy defaults
    'Fine (8×8×15×15)': [[-2.4, 2.4, 8], [-3.0, 3.0, 8], [-0.2095, 0.2095, 15], [-3.0, 3.0, 15]],
}

bin_results = {}
for name, bins in bin_configs.items():
    print(f'Training {name}...')
    p = QPolicy(state_bins=bins, n_actions=3, alpha=0.1, gamma=0.99, seed=42)
    r = train_q_learning(
        env, p, n_episodes=1000, epsilon_end=0.01,
        eval_interval=100, n_eval_episodes=20, verbose=False, seed=42
    )
    bin_results[name] = r
    qtable_mb = p.q_table.nbytes / 1024 / 1024
    print(f'  Best eval: {r["best_eval_reward"]:.1f}, Q-table: {p.q_table.size:,} entries ({qtable_mb:.2f} MB)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E74C3C', '#3498DB', '#2ECC71']

for (name, r), color in zip(bin_results.items(), colors):
    ax.plot(r['eval_episodes'], r['eval_rewards'], 'o-',
            label=name, color=color, linewidth=2, markersize=5)

ax.axhline(y=env.max_steps, color='gray', linestyle='--', alpha=0.7, label='Max possible')
ax.set_xlabel('Training Episode')
ax.set_ylabel('Mean Evaluation Reward (greedy, 20 eps)')
ax.set_title('Effect of Bin Resolution on Q-Learning Performance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 8: Summary and Limitations

### What Q-Learning Got Right
- **Interpretable**: Q-values have clear meaning — we can inspect what the agent learned
- **Convergence guarantees**: Under mild conditions, tabular Q-learning converges to Q*
- **Sample efficient per step**: Updates happen after every single transition

### Limitations on Continuous Control
- **Curse of dimensionality**: More state dimensions = exponentially more bins needed
- **Discretization loss**: Fine-grained control is impossible with discrete actions
- **Slow exploration**: High-dimensional Q-tables take many episodes to fill

### Next Steps
Tabular Q-learning is the foundation for much more powerful algorithms:
- **DQN (Deep Q-Network)**: Replace the Q-table with a neural network — handles continuous states naturally
- **Actor-Critic methods**: Combine policy-based and value-based approaches
- **Continuous Q-learning**: Use function approximation for continuous action spaces (e.g., DDPG, SAC)

---

## Exercises

1. **Learning rate sensitivity**: Try `alpha=0.01` and `alpha=0.5`. How does it affect convergence speed and stability?

2. **Discount factor**: Compare `gamma=0.9` vs `gamma=0.99`. Higher gamma values future rewards more — how does this change the learned behavior?

3. **More actions**: Try `n_actions=5` (5 discrete force levels). Does finer action resolution improve performance?

4. **Cosine annealing**: Switch `epsilon_schedule='cosine'` and compare convergence with exponential decay.

5. **Q-table inspection**: After training, examine `policy.get_q_values(state)` for states where the pole is strongly tilted left or right. Does the agent correctly prefer pushing in the opposing direction?